# 01 — Benchmark & entraînement (5 modèles + Optuna)

**Emplacement** : `02_modeles/01_benchmark_optuna.ipynb`

| Étape | Contenu |
|-------|--------|
| 1 | Vérifier les features (`01_donnees/04_pipeline_donnees.ipynb`) |
| 2 | Benchmark : reference · logistic · random_forest · lightgbm · mlp |
| 3 | Optuna (RF, LightGBM, MLP) — critère PR-AUC validation |
| 4 | Courbes ROC/PR + tableaux |
| 5 | Déploiement bundle webapp + SHAP / biais |

Équivalent : `PYTHONPATH=. python scripts/run_benchmark.py --optuna`

Prérequis macOS LightGBM : `brew install libomp`


## RF vs HistGB (rappel)

Sur le volume CISIA actuel → **Random Forest** en production.  
Grand volume hospitalier → **HistGB** plus efficace (voir détail : `04_entrainement_modele` §0 et webapp `/cisia/modeles`).

### Rappel des métriques

| Métrique | Signification |
|----------|---------------|
| **ROC-AUC** | Capacité à classer risque vs non-risque (0,5 = hasard, 1 = parfait) |
| **PR-AUC** | Performance sur la classe rare (réadmission) — **critère de choix** |
| **Rappel** | Part des vrais réadmissions détectées (priorité hôpital) |
| **Précision** | Part des alertes qui sont de vrais positifs |
| **F2** | Compromis rappel/précision (favorise le rappel) — sert au **seuil** |

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data.paths import ProjectPaths, get_project_root
from src.model.benchmark import model_catalog, run_benchmark
from src.model.bias import bias_table
from src.model.deploy import train_score
from src.model.explain import shap_top_features
from src.model.explain_patient import build_explication_risque_30j
from src.model.inference import predict_proba, threshold_for
from src.model.prepare import by_split, split_xy
from src.web.services import predict_sejour

paths = ProjectPaths(root=get_project_root())
plt.style.use("seaborn-v0_8-whitegrid")
print("Modèles benchmark :", [s.label for s in model_catalog()])

## 1. Vérifier les données

In [ ]:
FEATURES = {
    "score_sortie": paths.curated / "features_score_sortie.parquet",
    "score_tele": paths.curated / "features_score_tele.parquet",
}

for name, p in FEATURES.items():
    if not p.exists():
        raise FileNotFoundError(
            f"{p} absent — exécuter d'abord notebooks/pipeline_donnees.ipynb"
        )
    df = pd.read_parquet(p)
    print(f"{name}: {len(df)} séjours · prévalence {df['Readmission30j'].mean():.1%}")

## 2. Benchmark multi-modèles

In [ ]:
USE_OPTUNA = False  # True pour Optuna (RF, LightGBM, MLP) — plus long
OPTUNA_TRIALS = 25

reports = {}
for score_name, parquet in FEATURES.items():
    features = pd.read_parquet(parquet)
    reports[score_name] = run_benchmark(
        features,
        score_name=score_name,
        paths=paths,
        use_optuna=USE_OPTUNA,
        optuna_trials=OPTUNA_TRIALS,
    )
    print(f"\n=== {score_name} ===")
    print("Retenu :", reports[score_name]["retenu_label"], f"({reports[score_name]['retenu']})")
    print("Classement PR-AUC :", " → ".join(reports[score_name]["ranking_test_pr_auc"]))
    if reports[score_name].get("skipped"):
        print("Ignorés :", reports[score_name]["skipped"])
    print("HTML :", reports[score_name]["paths"]["html"])

## 3. Tableaux métriques (jeu test)

In [ ]:
def metrics_table(report: dict, split: str = "test") -> pd.DataFrame:
    rows = []
    for key in report["ranking_test_pr_auc"]:
        m = report["modeles"][key]
        s = m[split]
        rows.append({
            "modèle": m["label"],
            "clé": key,
            "PR-AUC": round(s["pr_auc"], 4),
            "ROC-AUC": round(s["roc_auc"], 4),
            "F2": round(s["f2"], 4),
            "Rappel": round(s["recall"], 4),
            "Précision": round(s["precision"], 4),
            "Seuil": round(s["threshold"], 2),
            "★": key == report["retenu"],
        })
    return pd.DataFrame(rows)

for score_name, report in reports.items():
    print(f"\n--- {score_name} ---")
    display(metrics_table(report))

## 4. Courbes ROC et PR

In [ ]:
COLORS = {
    "reference": "#79747e",
    "logistic": "#003735",
    "random_forest": "#693b24",
    "hist_gradient_boosting": "#515f74",
    "lightgbm": "#1b6b93",
    "mlp": "#ba1a1a",
    "hgb": "#515f74",
    "xgboost": "#ba1a1a",
}


def plot_curves(report: dict, *, split: str = "test") -> None:
    from IPython.display import display

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    for key in report["ranking_test_pr_auc"]:
        curves = report["modeles"][key].get("curves") or {}
        roc = curves.get(f"roc_{split}") or []
        pr = curves.get(f"pr_{split}") or []
        if not roc or not pr:
            continue
        label = report["modeles"][key]["label"]
        c = COLORS.get(key)
        axes[0].plot([p["x"] for p in roc], [p["y"] for p in roc], label=label, color=c, lw=2)
        axes[1].plot([p["x"] for p in pr], [p["y"] for p in pr], label=label, color=c, lw=2)
    axes[0].plot([0, 1], [0, 1], "--", color="#9a9a9a", lw=1, label="Hasard")
    axes[0].set_title(f"ROC — {split}")
    axes[0].set_xlabel("Faux positifs")
    axes[0].set_ylabel("Vrais positifs")
    axes[1].set_title(f"Courbe PR — {split}")
    axes[1].set_xlabel("Rappel")
    axes[1].set_ylabel("Précision")
    for ax in axes:
        ax.legend(fontsize=8)
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1.02)
    retenu = report.get("retenu_label") or report.get("retenu", "?")
    fig.suptitle(f"{report['score']} — retenu : {retenu}", fontweight="bold")
    fig.tight_layout()
    # Backend Agg (Docker / headless) : display() au lieu de plt.show()
    display(fig)
    plt.close(fig)


for report in reports.values():
    plot_curves(report)

## 5. Entraînement production (bundle webapp)

Entraîne logistic + HGB, déploie le gagnant dans `models/*_bundle.joblib`.

In [ ]:
train_reports = {}
for name, parquet in FEATURES.items():
    features = pd.read_parquet(parquet)
    train_reports[name] = train_score(
        features,
        score_name=name,
        paths=paths,
        use_optuna=USE_OPTUNA,
        optuna_trials=OPTUNA_TRIALS,
    )
    print(json.dumps({
        "score": name,
        "retenu": train_reports[name]["retenu"],
        "ranking": train_reports[name]["ranking_test_pr_auc"],
    }, ensure_ascii=False, indent=2))

## 6. SHAP et analyse de biais

In [ ]:
for name, parquet in FEATURES.items():
    features = pd.read_parquet(parquet)
    bundle = joblib.load(paths.models / f"{name}_bundle.joblib")
    X, y, split = split_xy(features)
    X_te, y_te = by_split(X, y, split, "test")
    retenu = train_reports[name]["retenu"]
    proba = predict_proba(bundle, retenu, X_te)
    thr = threshold_for(train_reports[name], retenu)
    pred = (proba >= thr).astype(int)
    bias = bias_table(X_te, y_te, pd.Series(pred, index=X_te.index))
    bias.to_csv(paths.models / f"{name}_biais.csv", index=False)
    if "coder" in bundle and "hgb" in bundle:
        shap_df = shap_top_features(bundle["coder"], bundle["hgb"], X)
        shap_df.to_csv(paths.models / f"{name}_shap.csv", index=False)
    print(f"\n=== {name} (déployé : {retenu}) ===")
    if "coder" in bundle and "hgb" in bundle:
        display(shap_df.head(10))
    display(bias.head(5))

## 7. Explication risque patient (démo webapp)

In [ ]:
features_sortie = pd.read_parquet(FEATURES["score_sortie"])
candidats = features_sortie.loc[features_sortie["Readmission30j"] == 1, "SejourID"]
sejour_demo = str(candidats.iloc[0]) if len(candidats) else str(features_sortie["SejourID"].iloc[0])

detail = predict_sejour(sejour_demo)
print(f"Séjour : {sejour_demo} · Proba {detail['proba_pct']}% · Seuil {detail['seuil_pct']}%")
print("\n", detail["explication_risque_30j"])
pd.DataFrame(detail["facteurs_risque_patient"])

## 8. Rapports HTML & webapp

- `models/benchmark/score_sortie_benchmark.html`
- `models/benchmark/score_tele_benchmark.html`

Redémarrer la webapp après l'étape 5 : `PYTHONPATH=. python scripts/run_webapp.py`

## Suite du parcours

**Prochaine étape :** [02 — Nettoyage & cohorte](02_nettoyage_donnees.ipynb)

| Étape | Notebook |
|-------|----------|
| → **02 — Nettoyage & cohorte** | [`02_nettoyage_donnees.ipynb`](02_nettoyage_donnees.ipynb) |
| Préparation features | [`03_preparation_modele.ipynb`](03_preparation_modele.ipynb) |
| Entraînement Optuna (cahier jury) | [`04_entrainement_modele.ipynb`](04_entrainement_modele.ipynb) |
| Sommaire | [`../00_guide/00_sommaire.ipynb`](../00_guide/00_sommaire.ipynb) |
